#AncientDoc OCR

1. Install related dependencies

In [1]:
!apt-get install tesseract-ocr
!pip install pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.


In [2]:
!wget https://github.com/tesseract-ocr/tessdata/raw/main/chi_tra_vert.traineddata
!mv chi_tra_vert.traineddata /usr/share/tesseract-ocr/4.00/tessdata/

--2026-01-05 21:51:18--  https://github.com/tesseract-ocr/tessdata/raw/main/chi_tra_vert.traineddata
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/tesseract-ocr/tessdata/main/chi_tra_vert.traineddata [following]
--2026-01-05 21:51:19--  https://raw.githubusercontent.com/tesseract-ocr/tessdata/main/chi_tra_vert.traineddata
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2368306 (2.3M) [application/octet-stream]
Saving to: ‘chi_tra_vert.traineddata’

chi_tra_vert.traine 100%[===================>]   2.26M  --.-KB/s    in 0.04s   

2026-01-05 21:51:19 (52.5 MB/s) - ‘chi_tra_vert.traineddata’ saved

In [3]:
import cv2
import numpy as np
import pytesseract
import matplotlib.pyplot as plt
from PIL import Image
import re

2. Build a baseline OCR pipeline

In [4]:
def baseline_ocr_pipeline(img_path):
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    denoised = cv2.medianBlur(gray, 3)

    binary = cv2.adaptiveThreshold(
        denoised, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 21, 10
    )
    projection = np.sum(binary, axis=0)
    threshold = np.max(projection) * 0.1

    in_line = False
    start_x = 0
    lines = []

    for x, val in enumerate(projection):
        if not in_line and val > threshold:
            in_line = True
            start_x = x
        elif in_line and val < threshold:
            in_line = False
            lines.append((start_x, x))
    # Read from right to left
    lines.sort(key=lambda x: x[0], reverse=True)

    results = []
    for i, (x1, x2) in enumerate(lines):
        roi = gray[:, x1:x2]
        text = pytesseract.image_to_string(
            roi, config="--oem 1 --psm 5 -l chi_tra_vert"
        ).strip()

        print(f"[Baseline] Column {i+1}: {text}")
        results.append(text)

    return results

baseline_output=baseline_ocr_pipeline('/content/page_3.png')

[Baseline] Column 1: 
[Baseline] Column 2: hi i|.  + |||| 婦 婦 婦 ! ll|| |
[Baseline] Column 3: _ 也 久 2 一 對 子 叉 生 蘆 從 之 其 生 【『
[Baseline] Column 4: 人 0 0 0 0 0 0 0 0
[Baseline] Column 5: vv G」. O 閨 ooqoOSGOSOOOOOOOOOO il 若
[Baseline] Column 6: 0( 了 看 人
[Baseline] Column 7: 一 一 用 和 及 你 信 信義 以 有 和 之 文 4 故 一 【
[Baseline] Column 8: 人 Kg KKKKKEKEKOWZ  5B/*  w- O 」OwvVvUqvOrUQqPqyqlT  ,.Il | S 了 V【  / | 凡 ..ww@@14u1IlTI)@ 人 p!uiiial:l 了 iiillili;,_iqPYPvYe 了 」L、Lk||i COCOmqm+qGeoGPmqmq''uuuuuouou 上 上 上 上 上 上 上 eu 上 eu :
[Baseline] Column 9: 主 二 - 虹 人 lt, 1
[Baseline] Column 10: _ 高 “人 儿 色 十 作怪 和 作客 仇 直 一 【
[Baseline] Column 11: 全 1 hisigSS|SSSvSS 放 點 sgVvgssgsEgGGGGgO OO 0:
[Baseline] Column 12: 一 2」 友 # 瑟 儿 / 閃 少 和 評估 人 民 生 多 法 一 【
[Baseline] Column 13: KiKkbhlhhbhhhgOVsO
[Baseline] Column 14: _ 未 可 如 有 仍 參 有 信 凡 有 召 也 有 毗 多 一 【
[Baseline] Column 15: iii. 首 kk
[Baseline] Column 16: 入 * h|;, kk  i | | | | | | | | | ) EL 0 :
[Baseline] Column 17: 還 k』 太 LU kh LEE LE ENLY LEE EL LIU LE ELHHFr PP

In [5]:
print(baseline_output)


['', 'hi i|.  + |||| 婦 婦 婦 ! ll|| |', '_ 也 久 2 一 對 子 叉 生 蘆 從 之 其 生 【『', '人 0 0 0 0 0 0 0 0', 'vv G」. O 閨 ooqoOSGOSOOOOOOOOOO il 若', '0( 了 看 人', '一 一 用 和 及 你 信 信義 以 有 和 之 文 4 故 一 【', "人 Kg KKKKKEKEKOWZ  5B/*  w- O 」OwvVvUqvOrUQqPqyqlT  ,.Il | S 了 V【  / | 凡 ..ww@@14u1IlTI)@ 人 p!uiiial:l 了 iiillili;,_iqPYPvYe 了 」L、Lk||i COCOmqm+qGeoGPmqmq''uuuuuouou 上 上 上 上 上 上 上 eu 上 eu :", '主 二 - 虹 人 lt, 1', '_ 高 “人 儿 色 十 作怪 和 作客 仇 直 一 【', '全 1 hisigSS|SSSvSS 放 點 sgVvgssgsEgGGGGgO OO 0:', '一 2」 友 # 瑟 儿 / 閃 少 和 評估 人 民 生 多 法 一 【', 'KiKkbhlhhbhhhgOVsO', '_ 未 可 如 有 仍 參 有 信 凡 有 召 也 有 毗 多 一 【', 'iii. 首 kk', '入 * h|;, kk  i | | | | | | | | | ) EL 0 :', '還 k』 太 LU kh LEE LE ENLY LEE EL LIU LE ELHHFr PP 111)1||j,|t|j||4 還', '_ 一 信人 之 公有 吱 擴 “ 馬 人 人, 級 可 一 【', 'MM 和 和 和 銜 放 馬上 >※※※wMM)DmklklukullkkhiktktkitkttktkEom 上 1 1wW 1 2 WU 0 歷 @T 王 |) |)Y | 〒_', '天 及 蟬 有 受 執 插 ‧ 儿 人 逐一 【『', '人 主 9 B※*,  》7|0|XY)|))))))))))|)))) 人 gg', '人 , 証 wwJw 唄 601l〒※wumˉmnˉ2 2 漸 4〈 旭 46*** 0 曙 》 光 ※※ 打 ˉ 沫 - 當 QOnIYI|IIwwYYIyYqq  mGi

3. Post-process OCR outputs

In [6]:
import re

def clean_chinese(text):
    return re.sub(r'[^\u4e00-\u9fa5]', '', text)

baseline_clean = clean_chinese("".join(baseline_output))
baseline_clean

'婦婦婦也久一對子叉生蘆從之其生人閨若了看人一一用和及你信信義以有和之文故一人了凡人了了上上上上上上上上主二虹人高人儿色十作怪和作客仇直一全放點一友瑟儿閃少和評估人民生多法一未可如有仍參有信凡有召也有毗多一首入還太還一信人之公有吱擴馬人人級可一和和和銜放馬上上歷王天及蟬有受執插儿人逐一人主人人証唄漸旭曙光打沫當旭太導玉人陸計迪過一隊公且了怕汽人限馬入座一和一及之人八務也多生多呈人字汐錠和子未失履夫流拿者一大困雪了人了肯怕食末冰介河媽表克胡太晟了螞上'

4. Evaluate performance using Character Error Rate (CER)

In [7]:
gt = "受之恒滿而又輒怕羨之甚有不及岌而削墳者捧之去何所勤廣岌五十餘種而意猶未已客曰後戯先生之岌何羡溢為不可窮若是則又曰異戯先生之岌受無何而出無何而受日轉徙轍轤於出與受之間岌乎且不得備主藏之職何秘之能為予曰茲乃所以為善藏也而亦不失祝夫藏金者"


In [8]:
def calculate_cer(reference, hypothesis):
    n, m = len(reference), len(hypothesis)
    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if reference[i - 1] == hypothesis[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = min(
                    dp[i - 1][j],     # deletion
                    dp[i][j - 1],     # insertion
                    dp[i - 1][j - 1]  # substitution
                ) + 1

    return dp[n][m] / n

In [9]:
baseline_cer = calculate_cer(gt, baseline_clean)
print(f"Baseline CER: {baseline_cer:.2%}")

Baseline CER: 182.76%


5. Implement and evaluate an optimized OCR pipeline

In [10]:
def optimized_ocr_pipeline(img_path):
    raw_img = cv2.imread(img_path)

    # Crop top margin to remove page noise
    h, w = raw_img.shape[:2]
    img = raw_img[int(h * 0.02):, :]

    # Convert to grayscale and enhance contrast
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)

    # Denoise
    gray = cv2.medianBlur(gray, 3)

    # Binarization
    _, binary = cv2.threshold(
        gray, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )

    # Remove vertical ruling lines
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
    remove_vertical = cv2.morphologyEx(
        binary, cv2.MORPH_OPEN, vertical_kernel, iterations=2
    )
    cnts = cv2.findContours(
        remove_vertical, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    cnts = cnts[0] if len(cnts) == 2 else cnts[1]
    for c in cnts:
        cv2.drawContours(binary, [c], -1, 0, 3)

    # Vertical projection with smoothing
    projection = np.sum(binary, axis=0)
    projection = np.convolve(projection, np.ones(10) / 10, mode='same')

    threshold = np.max(projection) * 0.15

    in_line = False
    start_x = 0
    lines = []

    for x, val in enumerate(projection):
        if not in_line and val > threshold:
            in_line = True
            start_x = x
        elif in_line and val < threshold:
            in_line = False
            lines.append((start_x, x))

    lines.sort(key=lambda x: x[0], reverse=True)

    # OCR
    results = []
    for i, (x1, x2) in enumerate(lines):
        roi = gray[:, x1:x2]
        text = pytesseract.image_to_string(
            roi, config="--oem 1 --psm 5 -l chi_tra_vert"
        )
        results.append(text)

        print(f"[Optimized] Column {i+1}: {text}")

    return results

optimized_output = optimized_ocr_pipeline('/content/page_3.png')

[Optimized] Column 1: 
[Optimized] Column 2: 久之 回 此 克 叉 輛 話 拱 之 竭 和 【『

[Optimized] Column 3: 和 及 爸 了 名 討 哇 儿 當 之 表 七 兩 一 【『

[Optimized] Column 4: 說 改色 十 八 全 各 包租 失 一 【

[Optimized] Column 5: 一 之 太 司 銷 蕉 才 生 之 信箱 和 入江 一 【

[Optimized] Column 6: 了 及 可 近 g 選 是 由 逮 司 翼 衣 -【

[Optimized] Column 7: 一 我 生 之 八 受 如 全 台 過 、 級 “可 一 『

[Optimized] Column 8: 一 看 及 司 霜 度 執 攝 狼人 次 及 <【

[Optimized] Column 9: 八 有 有 有 有 有 有 有 妾 Egg,

[Optimized] Column 10: 了 八 惠 有 也 汽 條 主 入 》 憲 【

[Optimized] Column 11: Mk ui 1L | ll 「l, 了 【L,

[Optimized] Column 12: 一 八 及 和 人 的 汪 馬 多 用 交 光 二 -【

[Optimized] Column 13: 入 eeGbgde 淋 暈 元 q:: | | | | | 肯 ˉ |〒 b 幅 | (e |7Q

[Optimized] Column 14: 一 信訪 地 了 未 了 腑 失 及 夫 沒 多 才 一 】【

[Optimized] Column 15: 
[Optimized] Column 16: 


In [11]:
optimized_clean = clean_chinese("".join(optimized_output))
optimized_clean

'久之回此克叉輛話拱之竭和和及爸了名討哇儿當之表七兩一說改色十八全各包租失一一之太司銷蕉才生之信箱和入江一了及可近選是由逮司翼衣一我生之八受如全台過級可一一看及司霜度執攝狼人次及八有有有有有有有妾了八惠有也汽條主入憲了一八及和人的汪馬多用交光二入淋暈元肯幅一信訪地了未了腑失及夫沒多才一'

In [12]:
optimized_cer = calculate_cer(gt, optimized_clean)
print(f"Optimized CER: {optimized_cer:.2%}")

Optimized CER: 109.48%


In [13]:
print(baseline_cer)
print(optimized_cer)
print(f"Improvement: {(baseline_cer - optimized_cer) / baseline_cer:.2%}")

1.8275862068965518
1.0948275862068966
Improvement: 40.09%


6. Test on more data

In [16]:
import os
import pandas as pd

with open('gt.txt', 'r', encoding='utf-8') as f:
    gt_lines = [line.strip() for line in f if line.strip()]
img_dir = '/content/'
files_with_size = []
for f in os.listdir(img_dir):
    if f.endswith('.png'):
        full_path = os.path.join(img_dir, f)
        files_with_size.append((f, os.path.getsize(full_path)))

files_with_size.sort(key=lambda x: x[1])
sorted_filenames = [f[0] for f in files_with_size]

results = []
target_nums = [2, 3, 4, 5, 6, 8, 9, 19, 20, 21, 22]

for n in target_nums:
    filename = f"page_{n}.png"
    img_path = os.path.join(img_dir, filename)

    try:
        idx = sorted_filenames.index(filename)
        if idx < len(gt_lines):
            gt_text = gt_lines[idx]
            b_out = baseline_ocr_pipeline(img_path)
            b_text = clean_chinese("".join(b_out))
            b_cer = calculate_cer(gt_text, b_text)

            o_out = optimized_ocr_pipeline(img_path)
            o_text = clean_chinese("".join(o_out))
            o_cer = calculate_cer(gt_text, o_text)

            results.append({
                "Page": filename,
                "Baseline_CER": b_cer,
                "Optimized_CER": o_cer,
                "Improvement": (b_cer - o_cer) / b_cer if b_cer > 0 else 0
            })
    except ValueError:
        print('No file')

[Baseline] Column 1: 二
[Baseline] Column 2: 
[Baseline] Column 3: EnvV CC〒T 0 mm
[Baseline] Column 4: 
[Baseline] Column 5: 
[Baseline] Column 6: 加 中 PS ku 還
[Baseline] Column 7: 人 0 于 KO) :
[Baseline] Column 8: 各 ma Dr DU
[Baseline] Column 9: 加  , PP 上 間 加
[Baseline] Column 10: 
[Baseline] Column 11: 人 點 I11 押 遇 嶄 kS  ! | | eps. 上 上 上 上 苛 liiDl 均 ※w s※'+omd 拉 lm 虐 辜 | ff 果 、% 四
[Baseline] Column 12: _ 天 人 AaA 和 先 法 之 化 儿 好 才 名 所 可 一 慧 一 【『
[Baseline] Column 13: 章 人 人 條 CC 二 人 : :
[Baseline] Column 14: 十 !! ※B「  Qq /〕/==mbQp QOQvpQO 濕 ˉ 6ˉGr7+ 0※※kl
[Baseline] Column 15: 
[Baseline] Column 16: 此 入 之 0 起 心 司 居 令 全 之 工控 及 必 一
[Baseline] Column 17: 集 gg
[Baseline] Column 18: 一 交 選 月 私人 表 光伏 守 光一 m『『
[Baseline] Column 19: 國 ll
[Baseline] Column 20: 一 一 上 次) 沒 > 生 條 沁 苛 到 生 多 生 (6 仇人 必 -m『【
[Baseline] Column 21: 人 00 ii i 學 ※)))))) 了 」tIIG72 | | | | III
[Baseline] Column 22: 
[Optimized] Column 1: ai 1

[Optimized] Column 2: 70 “LUC 畫 司 卻 信 多 人 和 還

[Optimized] Column 3: 一 到 7A 并 全 22 伯 入

In [17]:
df_results = pd.DataFrame(results)
print(df_results)
print(f"\nAverage Improvement: {df_results['Improvement'].mean():.2%}")

           Page  Baseline_CER  Optimized_CER  Improvement
0    page_2.png      1.916667       1.250000     0.347826
1    page_3.png      1.827586       1.094828     0.400943
2    page_4.png      1.461538       1.393162     0.046784
3    page_5.png      1.794643       1.821429    -0.014925
4    page_6.png      2.757009       1.233645     0.552542
5    page_8.png      1.752066       1.008264     0.424528
6    page_9.png      2.075000       1.300000     0.373494
7   page_19.png      1.672414       0.889655     0.468041
8   page_20.png      1.584906       1.003774     0.366667
9   page_21.png      1.522059       1.202206     0.210145
10  page_22.png      1.654412       0.948529     0.426667

Average Improvement: 32.75%
